# HKH Glacier — ablation rung runner (STW7088CEM)

## EVIDENCE CHECKLIST — capture BEFORE, DURING, and AFTER this run
The coursework requires screenshots of every experiment **showing the device**. Per rung:
1. **Start**: full-screen capture of this notebook running — Kaggle username (top right) AND the accelerator panel (right sidebar, GPU type) visible, plus the `nvidia-smi` cell output.
2. **Mid-training**: full-screen capture with the epoch log visible.
3. **End**: full-screen capture of the final metrics cell output.
4. Save as `evidence/rung_<RUNG>/{start,mid,end}.png` on your machine IMMEDIATELY.

A run without its screenshots is worth zero reproducibility marks.

In [ ]:
RUNG = "final"  # <-- set per run: a, b1, b2, c, d1, d2, d2b, e1, e2, f, final


In [ ]:
# Device evidence (screenshot this cell's output with the page chrome visible)
!nvidia-smi
!hostname
import torch, numpy, sys, datetime
print("python", sys.version)
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("numpy", numpy.__version__)
print("utc now:", datetime.datetime.now(datetime.UTC).isoformat())
# CUDA sanity: fail FAST if this torch build cannot launch kernels on this GPU
# (Kaggle's torch cu128 wheel dropped sm_60/P100 support -> run on T4, see kernel-metadata).
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device - do not burn a CPU session; fix accelerator")
x = torch.randn(64, 64, device="cuda")
torch.cuda.synchronize()
print("CUDA kernel launch OK:", float((x @ x).sum()), "on", torch.cuda.get_device_name(0))


In [ ]:
# Stage code from the attached code dataset (no internet needed).
# Robust to mount-layout quirks: locate train.py and a.yaml anywhere under /kaggle/input.
import shutil, os, zipfile, glob, pathlib

print("=== /kaggle/input tree (2 levels) ===")
for top in sorted(glob.glob("/kaggle/input/*")):
    print(top)
    for sub in sorted(glob.glob(top + "/*"))[:12]:
        print("   ", sub)

hits = list(pathlib.Path("/kaggle/input").rglob("train.py"))
assert hits, "train.py not found anywhere under /kaggle/input - is hkh-glacier-code attached?"
CODE_IN = str(hits[0].parent)
CODE = "/kaggle/working/code"
shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE, exist_ok=True)
for f in glob.glob(CODE_IN + "/*.py") + glob.glob(CODE_IN + "/*.txt"):
    shutil.copy(f, CODE)
yamls = list(pathlib.Path("/kaggle/input").rglob("a.yaml"))
if yamls:
    shutil.copytree(yamls[0].parent, CODE + "/configs")
else:
    zips = list(pathlib.Path("/kaggle/input").rglob("configs.zip"))
    assert zips, "no configs dir or configs.zip found under /kaggle/input"
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(CODE + "/configs")
print(sorted(os.listdir(CODE)), sorted(os.listdir(CODE + "/configs")))


In [ ]:
# Locate shards (robust: find the dir that contains train/ with npz files; zip fallback)
import os, zipfile, glob, shutil, pathlib
roots = [p.parent for p in pathlib.Path("/kaggle/input").rglob("train_stats.json")]
assert roots, "train_stats.json not found - is hkh-glacier-shards attached?"
DATA_IN = str(roots[0])
if glob.glob(DATA_IN + "/train/*.npz"):
    DATA = DATA_IN
else:
    DATA = "/kaggle/working/shards"
    os.makedirs(DATA, exist_ok=True)
    for zp in glob.glob(DATA_IN + "/*.zip"):
        name = os.path.basename(zp)[:-4]
        if not os.path.isdir(f"{DATA}/{name}"):
            with zipfile.ZipFile(zp) as z:
                z.extractall(f"{DATA}/{name}")
    for f in glob.glob(DATA_IN + "/*.json"):
        shutil.copy(f, DATA)
print("DATA root:", DATA)
for z in ("train", "val", "test", "east"):
    print(f"  {z}: {len(glob.glob(f'{DATA}/{z}/*.npz'))} shards")


In [ ]:
# Train this rung (screenshot mid-run). Checkpoints + metrics persist in /kaggle/working.
OUT = f"/kaggle/working/runs/{RUNG}"
!cd {CODE} && python train.py --config configs/{RUNG}.yaml --data {DATA} --out {OUT}


In [ ]:
# Validation metrics for the master ablation table (screenshot this output)
import glob
ckpt = sorted(glob.glob(f"{OUT}/*/best.pt"))[-1]
!cd {CODE} && python eval.py --ckpt {ckpt} --zone val --data {DATA} --examples 4
# Frozen split contract: test is evaluated EXACTLY ONCE, on the final model only;
# east is the OOD report, final model only. Both run automatically for RUNG=="final".
if RUNG == "final":
    print("\n=== ONE-SHOT TEST EVALUATION (contract clause 4) ===")
    !cd {CODE} && python eval.py --ckpt {ckpt} --zone test --data {DATA} --examples 6
    print("\n=== ONE-SHOT EAST OOD EVALUATION (contract clause 3) ===")
    !cd {CODE} && python eval.py --ckpt {ckpt} --zone east --data {DATA} --examples 6


In [ ]:
# Bundle run artifacts for download (metrics.jsonl, config, checkpoints, examples)
import shutil
shutil.make_archive(f"/kaggle/working/rung_{RUNG}_artifacts", "zip", OUT)
print("download: ", f"/kaggle/working/rung_{RUNG}_artifacts.zip")
